<a href="https://colab.research.google.com/github/Sayan1598/music-genre-classification-code/blob/main/music_genre_classification_for_GTZAN_and_ISMIR2004_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
data_path = '/content/drive/MyDrive/genres_original'

#For GTZAN dataset

In [ ]:

# Music Genre Classification (CNN + BiLSTM + BiGRU Hybrid)


import os
import random
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,
    Bidirectional, LSTM, GRU, Reshape, concatenate, Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import joblib


# 🎛 Step 1: Audio Augmentation

def augment_audio(y, sr):
    if random.random() < 0.5:
        y = librosa.effects.pitch_shift(y, sr, n_steps=random.uniform(-2, 2))
    if random.random() < 0.5:
        y = librosa.effects.time_stretch(y, rate=random.uniform(0.8, 1.2))
    if random.random() < 0.5:
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
    return y



# Step 2: Extract Mel Spectrogram Chunks

def extract_melspectrogram_chunks(y, sr, chunk_duration=4, overlap_duration=2, n_mels=256):
    chunk_samples = int(chunk_duration * sr)
    overlap_samples = int(overlap_duration * sr)
    num_chunks = max(1, int(np.ceil((len(y) - overlap_samples) / (chunk_samples - overlap_samples))))
    features = []

    for i in range(num_chunks):
        start = i * (chunk_samples - overlap_samples)
        end = start + chunk_samples
        chunk = y[start:end]

        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))

        mel_spec = librosa.feature.melspectrogram(
            y=chunk,
            sr=sr,
            n_fft=2048,
            hop_length=512,
            n_mels=n_mels,
            fmin=20,
            fmax=sr // 2
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = librosa.util.normalize(mel_spec_db)
        mel_spec_db = np.expand_dims(mel_spec_db, axis=-1)
        features.append(mel_spec_db)

    return features



# Step 3: Load Dataset

def load_dataset(data_dir, classes):
    X, y = [], []

    for genre in classes:
        genre_dir = os.path.join(data_dir, genre)
        print(f"Processing: {genre}")

        for file in os.listdir(genre_dir):
            if file.endswith('.wav'):
                file_path = os.path.join(genre_dir, file)
                try:
                    y_audio, sr = librosa.load(file_path, sr=None)
                    y_audio = augment_audio(y_audio, sr)
                    features = extract_melspectrogram_chunks(y_audio, sr)

                    for f in features:
                        X.append(f)
                        y.append(genre)

                except Exception as e:
                    # print(f" Skipping {file_path}: {e}")
                    print(" ")

    return np.array(X, dtype=np.float32), np.array(y)



# Step 4: Encode Labels + Split Dataset

def prepare_data(X, y):
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)
    y_onehot = to_categorical(y_encoded)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_onehot, test_size=0.2, random_state=42, stratify=y_onehot
    )
    return X_train, X_test, y_train, y_test, encoder


# Step 5: Build CNN + BiLSTM + BiGRU Model

def build_cnn_bilstm_bigru(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    # --- CNN Feature Extractor ---
    x = Conv2D(64, (3,3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)

    x = Conv2D(128, (3,3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.3)(x)

    x = Conv2D(256, (3,3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.4)(x)

    x = Reshape((-1, 256))(x)

    # Parallel BiLSTM and BiGRU branches
    bilstm_out = Bidirectional(LSTM(128, return_sequences=True))(x)
    bilstm_out = Dropout(0.4)(bilstm_out)
    bilstm_out = Bidirectional(LSTM(64, return_sequences=False))(bilstm_out)
    bilstm_out = Dropout(0.4)(bilstm_out)

    bigru_out = Bidirectional(GRU(128, return_sequences=True))(x)
    bigru_out = Dropout(0.4)(bigru_out)
    bigru_out = Bidirectional(GRU(64, return_sequences=False))(bigru_out)
    bigru_out = Dropout(0.4)(bigru_out)

    #Combine Both
    combined = concatenate([bilstm_out, bigru_out])

    # Dense Layers
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(combined)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    optimizer = Adam(learning_rate=1e-3)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model



# Step 6: Train the Model

def train_hybrid_model(data_dir, classes):
    X, y = load_dataset(data_dir, classes)
    print(f"\n Data Loaded: {X.shape[0]} samples, each shape: {X[0].shape}")

    X_train, X_test, y_train, y_test, encoder = prepare_data(X, y)
    input_shape = X_train.shape[1:]
    num_classes = y_train.shape[1]

    model = build_cnn_bilstm_bigru(input_shape, num_classes)
    model.summary()

    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=16,
        validation_data=(X_test, y_test),
        callbacks=[lr_scheduler, early_stop]
    )

    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"\nTest Accuracy: {test_acc:.3f}")

    return model, history, encoder, X_test, y_test



# Step 7: Evaluation

def evaluate_model(model, X_test, y_test, encoder, classes):
    y_pred = model.predict(X_test)
    y_true_labels = np.argmax(y_test, axis=1)
    y_pred_labels = np.argmax(y_pred, axis=1)

    cm = confusion_matrix(y_true_labels, y_pred_labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    print("\nClassification Report:\n")
    print(classification_report(y_true_labels, y_pred_labels, target_names=classes))

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(' Genre-wise Accuracy Heatmap (Hybrid Model)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    acc_per_genre = np.diag(cm_norm)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=classes, y=acc_per_genre, palette='mako')
    plt.title(' Per-Genre Accuracy (Hybrid Model)')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()



#  Step 8: Save Model + Encoder

def save_model_and_encoder(model, encoder):
    model.save("/content/music_genre_hybrid_model.h5")
    joblib.dump(encoder, "/content/label_encoder.pkl")
    print("\n Hybrid model and encoder saved successfully!")


# ============================================
# Step 9: Run Everything
# ============================================
data_dir = "/content/drive/MyDrive/genres_original"
classes = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']

model, history, encoder, X_test, y_test = train_hybrid_model(data_dir, classes)
evaluate_model(model, X_test, y_test, encoder, classes)
save_model_and_encoder(model, encoder)


In [ ]:
 Processing: blues
Processing: classical
Processing: country
Processing: disco
Processing: hiphop
Processing: jazz
/tmp/ipython-input-3040984207.py:90: UserWarning: PySoundFile failed. Trying audioread instead.
  y_audio, sr = librosa.load(file_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing: metal
Processing: pop
Processing: reggae
Processing: rock
 Data Loaded: 5009 samples, each shape: (256, 216, 1)
Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 216,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 216,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 216,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 108,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128, 108,  │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 108,  │     73,856 │ dropout[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 108,  │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 54,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64, 54,    │          0 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 54,    │    295,168 │ dropout_1[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 54,    │      1,024 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 27,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32, 27,    │          0 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 864, 256)  │          0 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 864, 256)  │    394,240 │ reshape[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 864, 256)  │    296,448 │ reshape[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 864, 256)  │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 864, 256)  │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │    164,352 │ dropout_3[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 128)       │    123,648 │ dropout_5[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 128)       │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ dropout_4[0][0],  │
│ (Concatenate)       │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    131,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 10)        │      2,570 │ dropout_8[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,618,698 (6.17 MB)

 Trainable params: 1,616,266 (6.17 MB)

 Non-trainable params: 2,432 (9.50 KB)

Epoch 1/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 116s 404ms/step - accuracy: 0.1424 - loss: 3.9967 - val_accuracy: 0.1377 - val_loss: 5.8799 - learning_rate: 0.0010
Epoch 2/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 133s 394ms/step - accuracy: 0.2176 - loss: 3.4110 - val_accuracy: 0.1707 - val_loss: 4.9511 - learning_rate: 0.0010
Epoch 3/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 142s 396ms/step - accuracy: 0.2620 - loss: 3.0768 - val_accuracy: 0.2555 - val_loss: 3.3960 - learning_rate: 0.0010
Epoch 4/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 142s 396ms/step - accuracy: 0.3204 - loss: 2.8265 - val_accuracy: 0.3054 - val_loss: 2.9723 - learning_rate: 0.0010
Epoch 5/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 142s 396ms/step - accuracy: 0.3674 - loss: 2.6657 - val_accuracy: 0.2645 - val_loss: 3.6192 - learning_rate: 0.0010
Epoch 6/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.4111 - loss: 2.4609 - val_accuracy: 0.2156 - val_loss: 3.6354 - learning_rate: 0.0010
Epoch 7/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 395ms/step - accuracy: 0.4845 - loss: 2.2399 - val_accuracy: 0.3882 - val_loss: 2.7235 - learning_rate: 0.0010
Epoch 8/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.5282 - loss: 2.0903 - val_accuracy: 0.3263 - val_loss: 3.0624 - learning_rate: 0.0010
Epoch 9/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.5788 - loss: 1.9258 - val_accuracy: 0.4142 - val_loss: 2.8924 - learning_rate: 0.0010
Epoch 10/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 389ms/step - accuracy: 0.6071 - loss: 1.8017 - val_accuracy: 0.3962 - val_loss: 2.8127 - learning_rate: 0.0010
Epoch 11/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.6152 - loss: 1.7682 - val_accuracy: 0.5240 - val_loss: 2.1424 - learning_rate: 0.0010
Epoch 12/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 390ms/step - accuracy: 0.6418 - loss: 1.6622 - val_accuracy: 0.3782 - val_loss: 3.1501 - learning_rate: 0.0010
Epoch 13/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.6960 - loss: 1.5325 - val_accuracy: 0.3483 - val_loss: 2.9988 - learning_rate: 0.0010
Epoch 14/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.6815 - loss: 1.5006 - val_accuracy: 0.4611 - val_loss: 2.4388 - learning_rate: 0.0010
Epoch 15/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step - accuracy: 0.7080 - loss: 1.4295
Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.7079 - loss: 1.4296 - val_accuracy: 0.3832 - val_loss: 3.0352 - learning_rate: 0.0010
Epoch 16/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.7576 - loss: 1.2851 - val_accuracy: 0.6327 - val_loss: 1.7562 - learning_rate: 5.0000e-04
Epoch 17/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.8032 - loss: 1.1406 - val_accuracy: 0.7784 - val_loss: 1.2032 - learning_rate: 5.0000e-04
Epoch 18/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.8165 - loss: 1.0924 - val_accuracy: 0.6018 - val_loss: 1.9127 - learning_rate: 5.0000e-04
Epoch 19/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 102s 407ms/step - accuracy: 0.8350 - loss: 1.0062 - val_accuracy: 0.6806 - val_loss: 1.4842 - learning_rate: 5.0000e-04
Epoch 20/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.8371 - loss: 1.0016 - val_accuracy: 0.5100 - val_loss: 2.3212 - learning_rate: 5.0000e-04
Epoch 21/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.8468 - loss: 0.9823 - val_accuracy: 0.8174 - val_loss: 1.0428 - learning_rate: 5.0000e-04
Epoch 22/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.8674 - loss: 0.8875 - val_accuracy: 0.7974 - val_loss: 1.0650 - learning_rate: 5.0000e-04
Epoch 23/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.8548 - loss: 0.9143 - val_accuracy: 0.7595 - val_loss: 1.3391 - learning_rate: 5.0000e-04
Epoch 24/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.8715 - loss: 0.8545 - val_accuracy: 0.8034 - val_loss: 1.1145 - learning_rate: 5.0000e-04
Epoch 25/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 102s 408ms/step - accuracy: 0.8683 - loss: 0.8409 - val_accuracy: 0.8403 - val_loss: 0.9013 - learning_rate: 5.0000e-04
Epoch 26/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.8613 - loss: 0.8681 - val_accuracy: 0.6976 - val_loss: 1.5106 - learning_rate: 5.0000e-04
Epoch 27/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.8761 - loss: 0.8123 - val_accuracy: 0.6956 - val_loss: 1.5231 - learning_rate: 5.0000e-04
Epoch 28/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.8887 - loss: 0.7769 - val_accuracy: 0.7864 - val_loss: 1.1903 - learning_rate: 5.0000e-04
Epoch 29/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step - accuracy: 0.8848 - loss: 0.7752
Epoch 29: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.8848 - loss: 0.7753 - val_accuracy: 0.6846 - val_loss: 1.5555 - learning_rate: 5.0000e-04
Epoch 30/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.9082 - loss: 0.7140 - val_accuracy: 0.8942 - val_loss: 0.7927 - learning_rate: 2.5000e-04
Epoch 31/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.9378 - loss: 0.6221 - val_accuracy: 0.8802 - val_loss: 0.8067 - learning_rate: 2.5000e-04
Epoch 32/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9377 - loss: 0.6107 - val_accuracy: 0.7345 - val_loss: 1.3912 - learning_rate: 2.5000e-04
Epoch 33/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.9313 - loss: 0.5872 - val_accuracy: 0.8014 - val_loss: 1.0701 - learning_rate: 2.5000e-04
Epoch 34/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.9377 - loss: 0.5864
Epoch 34: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9377 - loss: 0.5864 - val_accuracy: 0.8293 - val_loss: 0.9403 - learning_rate: 2.5000e-04
Epoch 35/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9519 - loss: 0.5377 - val_accuracy: 0.9232 - val_loss: 0.6011 - learning_rate: 1.2500e-04
Epoch 36/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.9647 - loss: 0.4837 - val_accuracy: 0.9251 - val_loss: 0.6284 - learning_rate: 1.2500e-04
Epoch 37/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9663 - loss: 0.4682 - val_accuracy: 0.9222 - val_loss: 0.6145 - learning_rate: 1.2500e-04
Epoch 38/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9680 - loss: 0.4677 - val_accuracy: 0.9192 - val_loss: 0.6070 - learning_rate: 1.2500e-04
Epoch 39/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.9648 - loss: 0.4715
Epoch 39: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9648 - loss: 0.4715 - val_accuracy: 0.9132 - val_loss: 0.6526 - learning_rate: 1.2500e-04
Epoch 40/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 395ms/step - accuracy: 0.9672 - loss: 0.4482 - val_accuracy: 0.9371 - val_loss: 0.5432 - learning_rate: 6.2500e-05
Epoch 41/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 142s 395ms/step - accuracy: 0.9686 - loss: 0.4396 - val_accuracy: 0.9491 - val_loss: 0.5203 - learning_rate: 6.2500e-05
Epoch 42/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 395ms/step - accuracy: 0.9705 - loss: 0.4241 - val_accuracy: 0.9511 - val_loss: 0.5256 - learning_rate: 6.2500e-05
Epoch 43/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 390ms/step - accuracy: 0.9739 - loss: 0.4176 - val_accuracy: 0.9242 - val_loss: 0.5659 - learning_rate: 6.2500e-05
Epoch 44/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9724 - loss: 0.4167 - val_accuracy: 0.9271 - val_loss: 0.5988 - learning_rate: 6.2500e-05
Epoch 45/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.9715 - loss: 0.4171 - val_accuracy: 0.9411 - val_loss: 0.5196 - learning_rate: 6.2500e-05
Epoch 46/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9752 - loss: 0.4110 - val_accuracy: 0.9271 - val_loss: 0.5343 - learning_rate: 6.2500e-05
Epoch 47/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9754 - loss: 0.3989 - val_accuracy: 0.9491 - val_loss: 0.5123 - learning_rate: 6.2500e-05
Epoch 48/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 391ms/step - accuracy: 0.9733 - loss: 0.3880 - val_accuracy: 0.9421 - val_loss: 0.5234 - learning_rate: 6.2500e-05
Epoch 49/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9725 - loss: 0.4009 - val_accuracy: 0.9291 - val_loss: 0.5711 - learning_rate: 6.2500e-05
Epoch 50/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9767 - loss: 0.3725 - val_accuracy: 0.9251 - val_loss: 0.5673 - learning_rate: 6.2500e-05
Epoch 51/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.9774 - loss: 0.3899
Epoch 51: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9774 - loss: 0.3899 - val_accuracy: 0.9311 - val_loss: 0.5603 - learning_rate: 6.2500e-05
Epoch 52/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9841 - loss: 0.3669 - val_accuracy: 0.9591 - val_loss: 0.4555 - learning_rate: 3.1250e-05
Epoch 53/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 98s 392ms/step - accuracy: 0.9821 - loss: 0.3633 - val_accuracy: 0.9551 - val_loss: 0.4616 - learning_rate: 3.1250e-05
Epoch 54/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 396ms/step - accuracy: 0.9826 - loss: 0.3487 - val_accuracy: 0.9521 - val_loss: 0.4795 - learning_rate: 3.1250e-05
Epoch 55/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.9811 - loss: 0.3544 - val_accuracy: 0.9561 - val_loss: 0.4619 - learning_rate: 3.1250e-05
Epoch 56/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step - accuracy: 0.9832 - loss: 0.3486
Epoch 56: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 394ms/step - accuracy: 0.9831 - loss: 0.3486 - val_accuracy: 0.9521 - val_loss: 0.4576 - learning_rate: 3.1250e-05
Epoch 57/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 102s 408ms/step - accuracy: 0.9824 - loss: 0.3518 - val_accuracy: 0.9541 - val_loss: 0.4772 - learning_rate: 1.5625e-05
Epoch 58/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 103s 409ms/step - accuracy: 0.9786 - loss: 0.3584 - val_accuracy: 0.9521 - val_loss: 0.4656 - learning_rate: 1.5625e-05
Epoch 59/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 393ms/step - accuracy: 0.9888 - loss: 0.3402 - val_accuracy: 0.9551 - val_loss: 0.4635 - learning_rate: 1.5625e-05
Epoch 60/60
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step - accuracy: 0.9819 - loss: 0.3457
Epoch 60: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.
251/251 ━━━━━━━━━━━━━━━━━━━━ 99s 392ms/step - accuracy: 0.9819 - loss: 0.3457 - val_accuracy: 0.9531 - val_loss: 0.4554 - learning_rate: 1.5625e-05
Restoring model weights from the end of the best epoch: 60.
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 147ms/step - accuracy: 0.9509 - loss: 0.4846

 Test Accuracy: 0.953
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 163ms/step

 Classification Report:

              precision    recall  f1-score   support

       blues       0.95      0.95      0.95       107
   classical       0.99      0.99      0.99       100
     country       0.95      0.91      0.93        96
       disco       0.97      0.96      0.96       100
      hiphop       0.93      0.95      0.94        84
        jazz       0.95      0.95      0.95       112
       metal       0.97      0.98      0.97        94
         pop       0.97      0.95      0.96       111
      reggae       0.99      0.97      0.98        93
        rock       0.87      0.92      0.90       105

    accuracy                           0.95      1002
   macro avg       0.95      0.95      0.95      1002
weighted avg       0.95      0.95      0.95      1002

/tmp/ipython-input-3040984207.py:224: UserWarning: Glyph 127932 (\N{MUSICAL SCORE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 127932 (\N{MUSICAL SCORE}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/tmp/ipython-input-3040984207.py:229: FutureWarning:

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=classes, y=acc_per_genre, palette='mako')
/tmp/ipython-input-3040984207.py:234: UserWarning: Glyph 127919 (\N{DIRECT HIT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/IPython/core/pylabtools.py:151: UserWarning: Glyph 127919 (\N{DIRECT HIT}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

 Hybrid model and encoder saved successfully!
